# 📂 Convert Election Files to CSV
Converts all `.txt` and `.xlsx` files under `data/` into `.csv`,
mirroring the folder structure into `data_csv/`.

```
data/Camera/19760620/camera-19760620.txt
→ data_csv/Camera/19760620/camera-19760620.csv
```

Handles:
- `;` separated txt files (older elections)
- `|` separated txt files (newer elections)
- `.xlsx` files with datetime cells
- Mixed encodings (auto-detected per file)

## Step 1 — Install & import

In [1]:
import subprocess, sys
for pkg in ['openpyxl', 'chardet']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

import csv
import chardet
import openpyxl
from pathlib import Path
from datetime import datetime

print('Ready.')

Ready.


## Step 2 — Configuration

In [2]:
SOURCE_ROOT  = Path('data')
DEST_ROOT    = Path('data_csv')
DATE_FORMAT  = '%Y-%m-%d'  # change to '%d/%m/%Y' for Italian format
SKIP_EXISTING = True       # set False to re-convert everything

print(f'Source : {SOURCE_ROOT.resolve()}')
print(f'Output : {DEST_ROOT.resolve()}')

Source : /Users/hajdumarcell/Documents/GitHub/BDA_project/data
Output : /Users/hajdumarcell/Documents/GitHub/BDA_project/data_csv


## Step 3 — Conversion functions

In [3]:
def detect_encoding(path: Path) -> str:
    """Detect file encoding. latin-1 is the safe fallback (decodes any byte)."""
    raw = path.read_bytes()
    result = chardet.detect(raw)
    encoding = result.get('encoding') or 'latin-1'
    confidence = result.get('confidence', 0)
    language = result.get('language', '')
    # chardet sometimes misdetects Windows-encoded Italian files as Hebrew ('he')
    # because the byte patterns overlap. In that case fall back to latin-1.
    if language in ('he', 'ar') or confidence < 0.5:
        encoding = 'latin-1'
    return encoding


def detect_separator(path: Path, encoding: str) -> str:
    """
    Sniff the separator by reading the first line and counting
    candidate delimiters. Returns the most frequent one.
    """
    try:
        with open(path, encoding=encoding, errors='replace') as f:
            first_line = f.readline()
    except Exception:
        return ';'

    candidates = [';', '|', ',', '\t']
    counts = {sep: first_line.count(sep) for sep in candidates}
    best = max(counts, key=counts.get)
    # Only use it if it actually appears; default to ;
    return best if counts[best] > 0 else ';'


def convert_txt(src: Path, dest: Path) -> int:
    """Convert a delimited .txt to .csv, auto-detecting encoding and separator."""
    encoding  = detect_encoding(src)
    separator = detect_separator(src, encoding)

    with open(src, encoding=encoding, newline='', errors='replace') as f_in:
        reader = csv.reader(f_in, delimiter=separator, quotechar='"')
        rows = list(reader)

    with open(dest, 'w', encoding='utf-8', newline='') as f_out:
        writer = csv.writer(f_out, delimiter=',', quotechar='"',
                            quoting=csv.QUOTE_MINIMAL)
        writer.writerows(rows)

    return len(rows)


def convert_xlsx(src: Path, dest: Path) -> int:
    """Convert the first sheet of an .xlsx to .csv, formatting dates cleanly."""
    wb = openpyxl.load_workbook(src, read_only=True, data_only=True)
    ws = wb.active

    rows = []
    for row in ws.iter_rows(values_only=True):
        cleaned = []
        for cell in row:
            if isinstance(cell, datetime):
                cleaned.append(cell.strftime(DATE_FORMAT))
            elif cell is None:
                cleaned.append('')
            else:
                cleaned.append(str(cell))
        rows.append(cleaned)
    wb.close()

    with open(dest, 'w', encoding='utf-8', newline='') as f_out:
        writer = csv.writer(f_out, delimiter=',', quotechar='"',
                            quoting=csv.QUOTE_MINIMAL)
        writer.writerows(rows)

    return len(rows)


print('Functions defined.')

Functions defined.


## Step 4 — Run conversion

In [4]:
files = sorted(
    p for p in SOURCE_ROOT.rglob('*')
    if p.suffix.lower() in ('.txt', '.xlsx')
)

if not files:
    print(f'No .txt or .xlsx files found under {SOURCE_ROOT}/')
else:
    converted, skipped, failed = 0, 0, 0

    for src in files:
        relative = src.relative_to(SOURCE_ROOT)
        dest     = DEST_ROOT / relative.with_suffix('.csv')

        if SKIP_EXISTING and dest.exists():
            skipped += 1
            continue

        dest.parent.mkdir(parents=True, exist_ok=True)

        try:
            if src.suffix.lower() == '.txt':
                n_rows = convert_txt(src, dest)
            else:
                n_rows = convert_xlsx(src, dest)
            print(f'  ✔  {src.name}  ({n_rows} rows)')
            converted += 1
        except Exception as e:
            print(f'  ❌  {src.name}  →  {e}')
            failed += 1

    print(f'\n✅ Done — {converted} converted, {skipped} skipped, {failed} failed.')

  ✔  camera-19480418.txt  (84193 rows)
  ✔  camera1948_preferenze.txt  (5607 rows)
  ✔  camera-19530607.txt  (91273 rows)
  ✔  camera1953_preferenze.txt  (6303 rows)
  ✔  camera-19580525.txt  (85752 rows)
  ✔  camera1958_preferenze.txt  (5980 rows)
  ✔  camera-19630428.txt  (74231 rows)
  ✔  camera1963_preferenze.txt  (5756 rows)
  ✔  camera-19680519.txt  (77772 rows)
  ✔  camera1968_preferenze.txt  (5837 rows)
  ✔  camera-19720507.txt  (90170 rows)
  ✔  camera1972_preferenze.txt  (6839 rows)
  ✔  camera-19760620.txt  (76948 rows)
  ✔  camera1976_preferenze.txt  (5854 rows)
  ✔  camera-19790603.txt  (95555 rows)
  ✔  camera1979_preferenze.txt  (7158 rows)
  ✔  camera-19830626.txt  (92863 rows)
  ✔  camera1983_preferenze.txt  (6855 rows)
  ✔  Camera-19870614.txt  (117346 rows)
  ✔  camera1987_preferenze.txt  (8632 rows)
  ✔  camera-19920405.txt  (138490 rows)
  ✔  camera1992_preferenze.txt  (9730 rows)
  ✔  Camera_19940327_Uninom_Cand&Contr.txt  (63825 rows)
  ✔  Camera_19940327_Uninom_